# Chapter 20: Reusable Workflows and Workflow Templates (notebook edition)

## Learning Objectives

- Read what crosses a reusable-workflow call
- Compare three ways to pass secrets
- Explain whose identity the OIDC token carries

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

GHA_MODE = os.environ.get("GHA_MODE", "fixture")  # "fixture" | "live"
GHA_REPO = os.environ.get("GHA_REPO", "Friend09/practice_git_intro_to_github_actions")
print("mode:", GHA_MODE, "| repo:", GHA_REPO)

## 1. The calls and their job names

Each called job is a real job named `<caller job> / <called job>`.

In [ ]:
import json
jobs = json.loads((ROOT/"fixtures"/"reusable_ch20_run.json").read_text())["jobs"]
for n, v in sorted(jobs.items()): print(f"{v['conclusion']:8} {n}")
assert len(jobs) == 8 and len(jobs) * 6 == 48   # mills if each is one billed Linux minute

## 2. What crosses the boundary

The called workflow saw the caller's name and event, repository variables and its inputs; it did not see the caller's `env`.

In [ ]:
r = jobs["call_local / build"]["reports"]
print("workflow:", r["workflow"], "| event:", r["event"], "| vars color:", r["color"], "| caller env:", r["caller_env"])
assert r["workflow"] == "[ch20 caller]" and r["color"] == "blue" and r["caller_env"] == "[]"

## 3. Secrets three ways

Length of the secret as seen by the called workflow.

In [ ]:
lens = {k: jobs[f"{k} / build"]["reports"]["secret_len"] for k in ("call_local", "call_inherit", "call_explicit_secret")}
print(lens)
assert lens == {"call_local": "0", "call_inherit": "23", "call_explicit_secret": "23"}

## 4. Outputs and defaults

Defaults fill omitted inputs; the output chain returned the summary.

In [ ]:
print(r["python"], r["run_tests"], jobs["consume_output"]["reports"])
assert (r["python"], r["run_tests"]) == ("3.12", "true")

## 5. Whose identity is the OIDC token?

The caller is in `workflow_ref`; the called workflow is in `job_workflow_ref`.

In [ ]:
o = jobs["oidc_in_called_workflow / claims"]["reports"]
print("job_workflow_ref:", o["oidc_job_workflow_ref"].split("/")[-1]); print("workflow_ref:    ", o["oidc_workflow_ref"].split("/")[-1])
assert "reusable-oidc" in o["oidc_job_workflow_ref"] and "ch20-caller" in o["oidc_workflow_ref"]

## Takeaways & Next Steps

- Pass inputs and named secrets; `env` does not cross.
- Grant permissions on the call job; failures are `startup_failure`.
- Next: Chapter 21, advanced techniques.

## Chapter Link

Read: `learning_modules/chapter_20_reusable_workflows.md`